In [0]:
%run /Workspace/Users/rana@ghazzi.com/Dev/Databricks_IBM/config_Parms

In [0]:
# Step 1: Fetches daily IBM stock data
def fetch_ibm_daily_data(api_key=api_key):
    url = f'https://www.alphavantage.co/query?function=TIME_SERIES_DAILY&symbol=IBM&apikey={api_key}'
    response = requests.get(url).json()
    data = response['Time Series (Daily)']
    dates = pd.DataFrame(data.keys())
    info = pd.DataFrame(data.values())
    last_updated = pd.DataFrame([response['Meta Data']['3. Last Refreshed']], columns=['Last_updated'])
    result = pd.concat([dates, info, last_updated], axis=1)
    result['Last_updated'] = result['Last_updated'].fillna(method='ffill')
    result = result.rename(columns={
        0           : 'Date',
        '1. open'   : 'Open',
        '2. high'   : 'High',
        '3. low'    : 'Low',
        '4. close'  : 'Close',
        '5. volume' : 'Volume'
    })
    result['ingested_at'] = pd.Timestamp.now()
    return result




In [0]:

def filter_new_rows_for_bronze(spark_df, table_path):
    """
    Filter only new rows to append based on the latest date value in the target table.
    """
    bronze_max_date = spark.sql(f"SELECT MAX(Date) FROM {table_path}").collect()[0][0]
    if bronze_max_date:
        new_rows = spark_df.filter(col('Date') > bronze_max_date)
    else:
        new_rows = spark_df
    return new_rows

In [0]:
from pyspark.sql.functions import col, current_timestamp
table_path='workspace.bronze.ibm'
result = fetch_ibm_daily_data()
spark_df = spark.createDataFrame(result)
new_rows=filter_new_rows_for_bronze(spark_df, table_path)



In [0]:
new_rows.display()

In [0]:
# Write parquet data here
new_rows.write.mode("append").parquet("/Volumes/workspace/bronze/landing_zone/ibm_landing/")

SELECT * FROM parquet.`/Volumes/workspace/bronze/landing_zone/ibm_landing/`